Just playing around with the APi

In [ ]:
%load_ext autoreload
%autoreload 2

import sys
sys.path.append('..')

import pandas as pd, sklearn, requests, matplotlib
# checking python path
print(sys.executable)
# checking versions 
print(pd.__version__, sklearn.__version__)

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload
/Users/biancajavier/github/biancacchino/bvb-title-predictor/.venv/bin/python
3.0.6 1.9.1


In [66]:
# fetching data, retrieves all matches from 2020 season of the  matchday
r = requests.get('https://api.openligadb.de/getmatchdata/bl1/2020/', timeout=10)
r.raise_for_status()
matches = r.json()
len(matches), matches

(306,
 [{'matchID': 58577,
   'matchDateTime': '2020-09-18T20:30:00',
   'timeZoneID': 'W. Europe Standard Time',
   'leagueId': 4442,
   'leagueName': '1. Fußball-Bundesliga 2020/2021',
   'leagueSeason': 2020,
   'leagueShortcut': 'bl1',
   'matchDateTimeUTC': '2020-09-18T18:30:00Z',
   'group': {'groupName': '1. Spieltag', 'groupOrderID': 1, 'groupID': 36486},
   'team1': {'teamId': 40,
    'teamName': 'FC Bayern München',
    'shortName': 'Bayern',
    'teamIconUrl': 'https://upload.wikimedia.org/wikipedia/commons/1/1f/Logo_FC_Bayern_M%C3%BCnchen_%282002%E2%80%932017%29.svg',
    'teamGroupName': None},
   'team2': {'teamId': 9,
    'teamName': 'FC Schalke 04',
    'shortName': 'S04',
    'teamIconUrl': 'https://upload.wikimedia.org/wikipedia/commons/9/97/FC_Schalke_04_Logo.png',
    'teamGroupName': None},
   'lastUpdateDateTime': '2020-09-20T16:15:35.617',
   'matchIsFinished': True,
   'matchResults': [{'resultID': 92144,
     'resultName': 'Endergebnis',
     'pointsTeam1': 8,


In [58]:
m = matches[0]
m['team2']['teamName'], m['group']['groupOrderID'], m['matchIsFinished']



('FC Schalke 04', 1, True)

In [60]:
for result in m['matchResults']:
    print(result['pointsTeam1'], result['pointsTeam2'])


def full_time(match):
    for result in match['matchResults']:
        # 2 is the score after full time, 1 is halftime
        if result['resultTypeID'] == 2:
            return result['pointsTeam1'], result['pointsTeam2']
    return None, None

full_time(m)
            

8 0
3 0


(8, 0)

In [61]:
home_goals, away_goals = full_time(m) # unpacks the tuple
row = {
    'season': m['leagueSeason'],
    'matchday': m['group']['groupOrderID'],
    'date': m['matchDateTime'],
    'home_team': m['team1']['teamName'],
    'away_team': m['team2']['teamName'],
    'home_goals': home_goals,
    'away_goals': away_goals,
    'finished': m['matchIsFinished']
}
row




{'season': 2020,
 'matchday': 1,
 'date': '2020-09-18T20:30:00',
 'home_team': 'FC Bayern München',
 'away_team': 'FC Schalke 04',
 'home_goals': 8,
 'away_goals': 0,
 'finished': True}

In [62]:
rows = []
for m in matches:
    home_goals, away_goals = full_time(m)
    rows.append({
        'season': m['leagueSeason'],
        'matchday': m['group']['groupOrderID'],
        'date': m['matchDateTime'],
        'home_team': m['team1']['teamName'],
        'away_team': m['team2']['teamName'],
        'home_goals': home_goals,
        'away_goals': away_goals,
        'finished': m['matchIsFinished']
    })

df = pd.DataFrame(rows)
df

,season,matchday,date,home_team,away_team,home_goals,away_goals,finished
0,2020,1,2020-09-18T20:30:00,FC Bayern München,FC Schalke 04,8,0,True
1,2020,1,2020-09-19T15:30:00,Eintracht Frankfurt,DSC Arminia Bielefeld,1,1,True
2,2020,1,2020-09-19T15:30:00,1. FC Köln,TSG Hoffenheim,2,3,True
3,2020,1,2020-09-19T15:30:00,VfB Stuttgart,SC Freiburg,2,3,True
4,2020,1,2020-09-19T15:30:00,1. FC Union Berlin,FC Augsburg,1,3,True
...,...,...,...,...,...,...,...,...
301,2020,34,2021-05-22T15:30:00,1. FC Köln,FC Schalke 04,1,0,True
302,2020,34,2021-05-22T15:30:00,VfB Stuttgart,DSC Arminia Bielefeld,0,2,True
303,2020,34,2021-05-22T15:30:00,1. FC Union Berlin,RB Leipzig,2,1,True
304,2020,34,2021-05-22T15:30:00,SV Werder Bremen,Borussia Mönchengladbach,2,4,True


In [64]:
df.shape 
# df.matchday.nunique()
# df.finished.all()

(306, 8)

In [ ]:
bvb_home = df[df.home_team == 'Borussia Dortmund'] # rows where bvb is the home team
(bvb_home.home_goals > bvb_home.away_goals).sum() # bvb number of wins home

np.int64(11)

In [ ]:
bvb_away = df[df.away_team == 'Borussia Dortmund'] # rows where bvb is the away team
(bvb_away.home_goals < bvb_away.away_goals).sum() # bvb number of wins away

np.int64(9)

In [114]:
bvb_draw_home = df[df.home_team == 'Borussia Dortmund']
(bvb_draw_home.home_goals == bvb_draw_home.away_goals).sum() # bvb number of draws
bvb_draw_away = df[df.away_team == 'Borussia Dortmund']
(bvb_draw_away.home_goals == bvb_draw_away.away_goals).sum() # bvb number of draws away



np.int64(2)

In [117]:
bvb_home = df[df.home_team == 'Borussia Dortmund']
bvb_away = df[df.away_team == 'Borussia Dortmund']

home_wins = (bvb_home.home_goals > bvb_home.away_goals).sum() # bvb number of wins home
away_wins = (bvb_away.home_goals < bvb_away.away_goals).sum() # bvb number of wins away
home_draws = (bvb_draw_home.home_goals == bvb_draw_home.away_goals).sum() # bvb number of draws home
away_draws = (bvb_draw_away.home_goals == bvb_draw_away.away_goals).sum() # bvb number of draws away

draws = home_draws + away_draws

bvb_points_2020 = ((home_wins + away_wins) * 3) + draws 
bvb_points_2020




np.int64(64)